<a href="https://colab.research.google.com/github/HereLiesAz/CueDetat/blob/main/ml/pocket_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cue d'Etat — Pocket Detector Training

Trains a **YOLOv8n** model on a billiards dataset and exports **TFLite FP16**
ready to drop into the `PocketDetector` interface in the app.

**Runtime:** GPU — Runtime → Change runtime type → T4 GPU.

The dataset may include multiple classes (balls, table, pockets, etc.).
The model is trained on all classes as-is. You only need to know which
class index corresponds to pockets — set that in Section 2 and the
smoke-test and Android integration will use it correctly.

In [23]:
from ultralytics import YOLO
import os

EPOCHS = 50
checkpoint_path = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights', 'last.pt')

if os.path.exists(checkpoint_path):
    model = YOLO(checkpoint_path)
    print(f'Resuming from checkpoint: {checkpoint_path}')
else:
    model = YOLO('yolov8n.pt')
    print('No checkpoint found. Starting fresh training with yolov8n.pt')

model.train(
    data=os.path.join(COMBINED_DIR, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=640,
    batch=32,
    project=PROJECT_DIR,
    name='pocket_detector',
    resume=os.path.exists(checkpoint_path)
)

Resuming from checkpoint: /content/drive/MyDrive/billiards_training/pocket_detector/weights/last.pt
Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/billiard

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import yaml

def merge_datasets(source_paths, target_path):
    master_classes = []
    for split in ['train', 'valid', 'test']:
        os.makedirs(os.path.join(target_path, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(target_path, split, 'labels'), exist_ok=True)

    for folder in source_paths:
        y_path = os.path.join(folder, 'data.yaml')
        if not os.path.exists(y_path):
            print(f'Warning: data.yaml not found in {folder}')
            continue

        with open(y_path, 'r') as f:
            cfg = yaml.safe_load(f)
            names = cfg.get('names', [])
            if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
            class_map = {i: (master_classes.index(n) if n in master_classes else (master_classes.append(n) or len(master_classes)-1)) for i, n in enumerate(names)}

        for split in ['train', 'valid', 'test']:
            src_split = split if os.path.exists(os.path.join(folder, split)) else 'val' if split == 'valid' else split
            s_img, s_lbl = os.path.join(folder, src_split, 'images'), os.path.join(folder, src_split, 'labels')
            if os.path.exists(s_img):
                for f in os.listdir(s_img): shutil.copy2(os.path.join(s_img, f), os.path.join(target_path, split, 'images', f))
            if os.path.exists(s_lbl):
                for f in os.listdir(s_lbl):
                    with open(os.path.join(s_lbl, f), 'r') as lf: lines = lf.readlines()
                    with open(os.path.join(target_path, split, 'labels', f), 'a') as lf:
                        for l in lines:
                            parts = l.split()
                            if parts: parts[0] = str(class_map[int(parts[0])]); lf.write(' '.join(parts) + '\n')

    with open(os.path.join(target_path, 'data.yaml'), 'w') as f:
        yaml.dump({'train': f'{target_path}/train/images', 'val': f'{target_path}/valid/images', 'test': f'{target_path}/test/images', 'nc': len(master_classes), 'names': master_classes}, f)
    return master_classes

# Point these to your ALREADY EXTRACTED folder paths
sources = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/Pool-Table-Detection-6',
    '/kaggle/input/cue-detat'
]

master_classes = merge_datasets(sources, COMBINED_DIR)
print(f'Merged Classes: {master_classes}')

In [ ]:
import os
import yaml

dataset_yaml = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml'

if os.path.exists(dataset_yaml):
    with open(dataset_yaml, 'r') as f:
        data = yaml.safe_load(f)

    print('--- Dataset Configuration ---')
    for key, value in data.items():
        if key not in ['train', 'val', 'test']:
            print(f'{key}: {value}')

    print('\n--- Split Statistics ---')
    for split in ['train', 'val', 'test']:
        # Some Yaml use 'val', directory is usually 'valid' or 'val'
        path_key = split if split in data else 'val' if split == 'valid' else None
        if path_key and path_key in data:
            img_path = data[path_key]
            if os.path.exists(img_path):
                count = len([f for f in os.listdir(img_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                print(f'{split.capitalize()} images: {count}')
            else:
                print(f'{split.capitalize()} path not found: {img_path}')
else:
    print(f'Error: {dataset_yaml} not found. Please ensure the merge step completed successfully.')

## 1. Install

In [ ]:
!pip install -q ultralytics roboflow kagglehub
import ultralytics
ultralytics.checks()

In [ ]:
from roboflow import Roboflow
import kagglehub

# 1. Download Dataset A (Roboflow)
rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("leonardo-wijaya-bdcih").project("pool-table-detection")
dataset_a = project.version(6).download("yolov8")

# 2. Download Dataset B (Kaggle)
kaggle_path = kagglehub.dataset_download("hereliesaz/cue-detat")

print(f"Data sources ready at: {dataset_a.location} and {kaggle_path}")

In [ ]:
import zipfile
import os
import yaml

# List of zip files to extract
datasets_zips = [
    '/content/CueTor Billiards Training.v8i.yolov8.zip',
    '/content/Pool Table Detection.v6i.yolov8.zip'
]

def extract_and_inspect(zip_path):
    folder_name = os.path.basename(zip_path).replace('.zip', '').replace(' ', '_')
    extract_to = f'/content/{folder_name}'

    print(f'--- Processing: {os.path.basename(zip_path)} ---')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        yaml_path = os.path.join(extract_to, 'data.yaml')
        if os.path.exists(yaml_path):
            with open(yaml_path, 'r') as f:
                data = yaml.safe_load(f)
                names = data.get('names', [])
                print(f'Location: {extract_to}')
                print(f'Classes: {names}')
        return extract_to
    except Exception as e:
        print(f'Error extracting {zip_path}: {e}')
        return None

# Extract zip files and keep track of folders
extracted_paths = []
for ds in datasets_zips:
    if os.path.exists(ds):
        path = extract_and_inspect(ds)
        if path: extracted_paths.append(path)
    else:
        print(f'File not found: {ds}')

# Add the Roboflow downloaded folder to the list
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f'Added Roboflow dataset: {roboflow_dir}')

In [ ]:
import glob
import os

# Search for any zip files in the content directory
found_zips = glob.glob('/content/*.zip')
print(f'Found zip files: {found_zips}')

# Filter for relevant datasets and update the list for cell 5761cd6d
datasets_zips = [f for f in found_zips if 'Billiards' in f or 'Pool' in f]

if not datasets_zips:
    print('No matching zip files found. Please upload your dataset zips to the /content folder.')
else:
    print(f'Updated dataset list to: {datasets_zips}')

# Re-run the inspection/extraction logic if paths are found
for ds in datasets_zips:
    path = extract_and_inspect(ds)
    if path and path not in extracted_paths:
        extracted_paths.append(path)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Define core paths
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

In [ ]:
!pip install -q roboflow
import os
from roboflow import Roboflow
from google.colab import drive

if not os.path.exists('/content/drive'): drive.mount('/content/drive')

# Target path on Drive
balls_path = '/content/drive/MyDrive/billiards_training/datasets/balls_dataset'
os.makedirs(balls_path, exist_ok=True)
%cd {balls_path}

rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("wrckmn1").project("pool-balls-yolov11m")
version = project.version(14)
dataset = version.download("yolov8")

print(f"Balls dataset downloaded to Drive: {balls_path}")

In [ ]:
import os

# Manually re-populate extracted_paths based on existing directories
extracted_paths = []

# Check for the Roboflow folder
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f"Found Roboflow dataset: {roboflow_dir}")

# Check for any other folders created by extraction (excluding standard Colab folders)
for item in os.listdir('/content'):
    full_path = os.path.join('/content', item)
    if os.path.isdir(full_path) and item not in ['.config', 'drive', 'sample_data', 'combined_dataset']:
        if full_path not in extracted_paths:
            extracted_paths.append(full_path)
            print(f"Found extracted dataset: {full_path}")

print(f"Final list of datasets to merge: {extracted_paths}")

In [ ]:
import kagglehub
path = kagglehub.dataset_download("hereliesaz/cue-detat")

In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

In [ ]:
import os

# Verification Check
sources = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/Pool-Table-Detection-6',
    '/kaggle/input/cue-detat'
]

print('--- Pre-Merge Validation ---')
for path in sources:
    if os.path.exists(path):
        y_path = os.path.join(path, 'data.yaml')
        has_yaml = os.path.exists(y_path)
        print(f'✅ Found: {path}')
        print(f'   - data.yaml: {"Found" if has_yaml else "MISSING"}')
    else:
        print(f'❌ NOT FOUND: {path}')

# Check if combined directory is ready
print(f'\nTarget directory for merging: {COMBINED_DIR}')

In [ ]:
from ultralytics import YOLO
import os

# Training Configuration
EPOCHS = 50
checkpoint_path = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights', 'last.pt')

if os.path.exists(checkpoint_path):
    model = YOLO(checkpoint_path)
    print(f'Resuming from checkpoint: {checkpoint_path}')
else:
    model = YOLO('yolov8n.pt')
    print('No checkpoint found. Starting fresh training with yolov8n.pt')

# To make this 'pausable', we ensure save=True and set a save_period.
# You can stop the cell execution at any time and re-run this cell to resume.
model.train(
    data=os.path.join(COMBINED_DIR, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=640,
    batch=32,
    project=PROJECT_DIR,
    name='pocket_detector',
    save=True,
    save_period=5, # Saves a checkpoint every 5 epochs
    resume=os.path.exists(checkpoint_path)
)

In [ ]:
import glob
import os

# Search for any 'best.pt' files in the training directory
search_path = "/content/drive/MyDrive/billiards_training/**/best.pt"
found_weights = glob.glob(search_path, recursive=True)

if found_weights:
    print("Found weights at:")
    for w in sorted(found_weights):
        print(f"- {w}")

    # Pick the most recent one (usually the one with the highest number or latest timestamp)
    latest_weights = sorted(found_weights)[-1]
    print(f"\nRecommended weights for export: {latest_weights}")
else:
    print("No 'best.pt' files found. Please ensure training completed successfully.")

In [ ]:
import os
from roboflow import Roboflow
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

    # Target path on Drive
    cuetor_path = '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8'
    os.makedirs(cuetor_path, exist_ok=True)
    %cd {cuetor_path}

    # Use Universe API approach for public datasets
    rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
    project = rf.universe("cuetor").project("cuetor-billiards-training")
    version = project.version(8)
    dataset = version.download("yolov8")

    print(f"CueTor dataset downloaded to Drive: {cuetor_path}")

In [ ]:
import os
import shutil
import yaml
import kagglehub

# 1. Download Kaggle dataset
kaggle_path = kagglehub.dataset_download("hereliesaz/cue-detat")

# 2. Source directories
extracted_paths = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/pool-balls-yolov11m-14',
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    kaggle_path
]

combined_dir = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset'
if os.path.exists(combined_dir): shutil.rmtree(combined_dir)
os.makedirs(combined_dir, exist_ok=True)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(combined_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(combined_dir, split, 'labels'), exist_ok=True)

master_classes = []

print("Merging ALL datasets (Drive + Kaggle)...")
for folder in extracted_paths:
    if not os.path.exists(folder):
        print(f"Warning: {folder} not found.")
        continue

    y_path = os.path.join(folder, 'data.yaml')
    if not os.path.exists(y_path):
        subdirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
        for sd in subdirs:
            if os.path.exists(os.path.join(folder, sd, 'data.yaml')):
                folder = os.path.join(folder, sd)
                y_path = os.path.join(folder, 'data.yaml')
                break

    if not os.path.exists(y_path): continue

    with open(y_path, 'r') as f:
        cfg = yaml.safe_load(f)
        names = cfg.get('names', [])
        if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]

        class_map = {}
        for i, name in enumerate(names):
            if name not in master_classes: master_classes.append(name)
            class_map[i] = master_classes.index(name)

    for split in ['train', 'valid', 'test']:
        src_split = split if os.path.exists(os.path.join(folder, split)) else 'val' if split == 'valid' else split
        s_img, s_lbl = os.path.join(folder, src_split, 'images'), os.path.join(folder, src_split, 'labels')

        if os.path.exists(s_img):
            for f in os.listdir(s_img):
                shutil.copy2(os.path.join(s_img, f), os.path.join(combined_dir, split, 'images', f))
        if os.path.exists(s_lbl):
            for f in os.listdir(s_lbl):
                with open(os.path.join(s_lbl, f), 'r') as lf: lines = lf.readlines()
                with open(os.path.join(combined_dir, split, 'labels', f), 'w') as lf:
                    for l in lines:
                        p = l.split()
                        if p:
                            p[0] = str(class_map.get(int(p[0]), p[0]))
                            lf.write(' '.join(p) + '\n')

with open(os.path.join(combined_dir, 'data.yaml'), 'w') as f:
    yaml.dump({'train': f'{combined_dir}/train/images', 'val': f'{combined_dir}/valid/images', 'test': f'{combined_dir}/test/images', 'nc': len(master_classes), 'names': master_classes}, f)

print(f"All datasets combined on Drive: {combined_dir}")
print(f"Final Classes: {master_classes}")

In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

In [ ]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

In [ ]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

## 2. Configuration

Fill these in, then run all cells.

## Final Pipeline: Pocket Detector Training

This consolidated pipeline handles environment setup, multi-source dataset extraction, merging, and YOLOv8 training with TFLite export.

## 4. Train

In [ ]:
from ultralytics import YOLO
import os
import torch

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 16 # Adjusted for T4 RAM stability
DATASET_YAML = "/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Find the index for 'pool-table-hole' dynamically
try:
    pocket_idx = master_classes.index('pool-table-hole')
    POCKET_CLASS_IDX = [pocket_idx]
    print(f"Detected pocket class index: {POCKET_CLASS_IDX}")
except (NameError, ValueError):
    POCKET_CLASS_IDX = [1]
    print("Defaulting POCKET_CLASS_IDX to [1]")

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolov8n.pt")

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    name="pocket_detector",
    project=PROJECT_DIR,
    save=True
)

print(f"Training complete. Weights are in: {results.save_dir}")

In [ ]:
from ultralytics import YOLO
import os

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = "/content/combined_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Ensure the base directory exists before training starts
os.makedirs(PROJECT_DIR, exist_ok=True)

# Resume Logic: Check if 'last.pt' exists in the weights folder
checkpoint_path = os.path.join(PROJECT_DIR, "pocket_detector", "weights", "last.pt")
resume_training = os.path.exists(checkpoint_path)

if resume_training:
    print(f"Found checkpoint at {checkpoint_path}. Resuming training...")
    model = YOLO(checkpoint_path)
else:
    print("No checkpoint found. Starting fresh training with yolov8n.pt")
    model = YOLO("yolov8n.pt")

# Train/Resume the model
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=20,
    name="pocket_detector",
    project=PROJECT_DIR,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    save=True,
    save_period=5,
    resume=resume_training
)

print(f"Training complete or paused. Weights are in: {results.save_dir}")

In [ ]:
import os

# --- GitHub Configuration ---
# IMPORTANT: Fill these in with your actual details!
GITHUB_TOKEN = "your_token_here"
GITHUB_USER = "your_username"
GITHUB_REPO = "your_repo_name"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Navigate to your drive project folder
if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}

    # Initialize git if not already done
    if not os.path.exists('.git'):
        !git init
        !git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git
        !git branch -M main

    # Configure user
    !git config --global user.email "you@example.com"
    !git config --global user.name "{GITHUB_USER}"

    # Add, commit and push
    !git add .
    !git commit -m "Update training weights and logs"
    !git push -u origin main
    print("Changes pushed to GitHub.")
else:
    print(f"Error: {PROJECT_DIR} not found. Ensure training has started and created this directory.")

## 5. Validate

In [ ]:
import os
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=DATASET_YAML)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print()
if metrics.box.map50 > 0.85:
    print("✓ mAP50 > 0.85 — good to export")
else:
    print("⚠ mAP50 below 0.85 — consider more epochs or more data")

## 6. Export to TFLite FP16

In [ ]:
from ultralytics import YOLO
import os
import shutil

# Define paths
weights_path = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights', 'best.pt')
drive_export_dir = '/content/drive/MyDrive/billiards_training/exports'
os.makedirs(drive_export_dir, exist_ok=True)

# Load model
model = YOLO(weights_path)

# Export to TFLite FP16
print("Exporting to TFLite...")
export_result = model.export(
    format='tflite',
    imgsz=640,
    half=True,
    nms=True
)

# The export method returns the path to the exported file/folder
target_tflite = os.path.join(drive_export_dir, 'pocket_detector_fp16.tflite')

if export_result.endswith('.tflite'):
    shutil.copy2(export_result, target_tflite)
else:
    # If it exported to a directory, find the .tflite file inside
    tflite_files = [f for f in os.listdir(export_result) if f.endswith('.tflite')]
    if tflite_files:
        shutil.copy2(os.path.join(export_result, tflite_files[0]), target_tflite)

print(f'Export complete! TFLite file saved to Drive: {target_tflite}')

## 7. Smoke-test

Shows detections for the pocket class only (filtered by `POCKET_CLASS_IDX`).

In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d = interp.get_input_details()
out_d = interp.get_output_details()
SZ    = int(inp_d[0]['shape'][1])
dtype = inp_d[0]['dtype']
print("Input:", inp_d[0]['shape'], dtype)

def infer(path):
    bgr = cv2.imread(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], w, h

yaml_dir = os.path.dirname(DATASET_YAML) + "/"
imgs = (glob.glob(yaml_dir + 'valid/images/*.jpg') +
        glob.glob(yaml_dir + 'valid/images/*.png'))[:4]

fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(5 * max(len(imgs), 1), 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        # d = [x1, y1, x2, y2, conf, class_idx]  (with baked NMS)
        conf = float(d[4]) if len(d) >= 5 else 0
        cls  = int(d[5])   if len(d) >= 6 else 0
        if conf >= CONF_THRESHOLD and cls in POCKET_CLASS_IDX:
            x1,y1,x2,y2 = (int(d[0]*w/SZ), int(d[1]*h/SZ),
                            int(d[2]*w/SZ), int(d[3]*h/SZ))
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(draw, f"{conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')

plt.tight_layout(); plt.show()

## 8. Download

In [ ]:
import shutil
import os

# Verify weights directory on Drive
train_run_weights = '/content/drive/MyDrive/billiards_training/pocket_detector/weights'
backup_dir = '/content/drive/MyDrive/billiards_training/snapshots_backup'

if os.path.exists(train_run_weights):
    if os.path.exists(backup_dir): shutil.rmtree(backup_dir)
    shutil.copytree(train_run_weights, backup_dir)
    print(f'All snapshots (best.pt, last.pt, etc.) backed up to: {backup_dir}')
else:
    print(f'Warning: Weights directory not found at {train_run_weights}. Check PROJECT_DIR.')

# Task
Clean and consolidate the billiards pocket detector training pipeline. Install `ultralytics`, `roboflow`, and `kagglehub`; mount Google Drive; and define `PROJECT_DIR` and `COMBINED_DIR`. Download the "cue-detat" dataset from kagglehub and save it to "/content/drive/MyDrive/billiards_training/datasets/". Perform a consolidated merge of the four datasets ("pool-balls", "Pool-Table-Detection", "CueTor", and "Cue-Detat") from their respective Drive/Kaggle paths into `COMBINED_DIR`, generating a master `data.yaml`. Dynamically identify the class index for 'pool-table-hole' as `POCKET_CLASS_IDX`. Execute a pausable YOLOv8n training session with checkpoints saved to Drive. Finally, validate the model, export it to TFLite FP16 format in the Drive exports folder, and remove all redundant cells from the notebook.

## Consolidate Setup

### Subtask:
Install required libraries, mount Google Drive, and define global path variables for the project and combined dataset.


**Reasoning**:
I will consolidate the setup by installing dependencies, mounting Google Drive, and defining the necessary project path variables as requested in the subtask instructions.



In [ ]:
!pip install -q ultralytics roboflow kagglehub

import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define global path variables
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# Ensure directories exist
os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

print(f'Project Directory: {PROJECT_DIR}')
print(f'Combined Dataset Directory: {COMBINED_DIR}')

## Download Cue d'Etat to Drive

### Subtask:
Download the 'cue-detat' dataset from Kaggle and move it to the persistent project directory on Google Drive.


**Reasoning**:
I will add a code block to download the 'cue-detat' dataset from Kaggle and transfer it to the persistent directory on Google Drive as instructed.



In [ ]:
import kagglehub
import shutil
import os

# 1. Download the dataset from Kaggle
kaggle_src_path = kagglehub.dataset_download('hereliesaz/cue-detat')

# 2. Define the destination path on Google Drive
drive_dest_path = '/content/drive/MyDrive/billiards_training/datasets/cue-detat'

# 3. Move or copy the dataset to Google Drive
# Remove existing directory if it exists to avoid errors during copy
if os.path.exists(drive_dest_path):
    shutil.rmtree(drive_dest_path)

shutil.copytree(kaggle_src_path, drive_dest_path)

# 4. Verify the transfer
if os.path.exists(drive_dest_path):
    print(f'Dataset successfully moved to: {drive_dest_path}')
    print(f'Files in directory: {os.listdir(drive_dest_path)}')
else:
    print(f'Error: Failed to move dataset to {drive_dest_path}')

# Task
None

# Task
Clean and consolidate the billiards pocket detector training pipeline. Start by installing `ultralytics`, `roboflow`, and `kagglehub`, mounting Google Drive, and defining project variables `PROJECT_DIR` and `COMBINED_DIR`.

The core task involves preparing the data by:
1. Extracting the reuploaded zip files in `/content` (specifically "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip") and persisting them to Google Drive.
2. Downloading the "cue-detat" dataset from `kagglehub` and moving it to "/content/drive/MyDrive/billiards_training/datasets/cue-detat".
3. Executing a consolidated merge of all four datasets ("pool-balls", "Pool-Table-Detection", "CueTor", and "Cue-Detat") from their persistent Drive paths into `COMBINED_DIR`, generating a master `data.yaml`.

Following data preparation:
1. Dynamically detect the class index for 'pool-table-hole' from the merged dataset and define it as `POCKET_CLASS_IDX`.
2. Configure and run a pausable YOLOv8n training session that supports resuming from Drive checkpoints and saves weights periodically.
3. Validate the best weights and export the model to TFLite FP16 format in the Drive exports folder.
4. Clean the notebook by removing redundant cells and reordering the functional cells into a logical pipeline.

## Extract and Persist Reuploaded Data

### Subtask:
Identify the reuploaded zip files in /content, extract their contents, and move them to the persistent datasets directory on Google Drive.


**Reasoning**:
I will create a code block to extract the three specific zip files from the /content directory and move their contents to the persistent project directory on Google Drive as instructed.



In [ ]:
import zipfile
import shutil
import os

# 1. Define list of zip files in /content
dataset_zips = [
    '/content/Pool Table Detection.v6i.yolov8.zip',
    '/content/CueTor Billiards Training.v8i.yolov8.zip',
    '/content/pool-balls-yolov11m.v14i.yolov8.zip'
]

# Map zip filenames to a cleaner folder name for Drive
zip_to_folder = {
    'Pool Table Detection.v6i.yolov8.zip': 'Pool-Table-Detection-6',
    'CueTor Billiards Training.v8i.yolov8.zip': 'CueTor-Billiards-Training-8',
    'pool-balls-yolov11m.v14i.yolov8.zip': 'pool-balls-yolov11m-14'
}

print('--- Extracting and Persisting Datasets ---')

for zip_path in dataset_zips:
    if not os.path.exists(zip_path):
        print(f'File not found: {zip_path}')
        continue

    zip_name = os.path.basename(zip_path)
    folder_name = zip_to_folder[zip_name]

    # 2. Extract contents to a temporary directory in /content
    temp_extract_path = f'/content/temp_{folder_name}'
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_extract_path)

    # 3. Define target directory in DATASETS_BASE (Google Drive)
    drive_target_path = os.path.join(DATASETS_BASE, folder_name)

    # 4. Move extracted folder to persistent location on Google Drive
    if os.path.exists(drive_target_path):
        shutil.rmtree(drive_target_path)

    shutil.move(temp_extract_path, drive_target_path)

    # 5. Print final paths to verify
    print(f'✅ Extracted and moved {zip_name} to:')
    print(f'   {drive_target_path}')

# Task
Clean and consolidate the billiards pocket detector training pipeline by first installing `ultralytics`, `roboflow`, and `kagglehub`, mounting Google Drive, and defining project variables `PROJECT_DIR` and `COMBINED_DIR`.

The data preparation phase involves:
1. Extracting the three reuploaded zip files in `/content` (specifically "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip") using robust extraction logic and persisting the resulting folders to Google Drive at "/content/drive/MyDrive/billiards_training/datasets/".
2. Downloading the "cue-detat" dataset from `kagglehub` and moving it to the persistent Drive path "/content/drive/MyDrive/billiards_training/datasets/cue-detat".
3. Executing a consolidated four-way merge of "pool-balls", "Pool-Table-Detection", "CueTor", and "Cue-Detat" from their persistent Drive locations into `COMBINED_DIR`, generating a unified `data.yaml`.

Post-data preparation:
1. Dynamically identify the class index for 'pool-table-hole' from the merged master class list and define it as `POCKET_CLASS_IDX`.
2. Configure and execute a pausable YOLOv8n training session that supports resuming from Drive checkpoints (`last.pt`) and utilizes a `save_period` for periodic progress saving.
3. Validate the final model using the best weights and export it to TFLite FP16 format in the Drive exports folder.
4. Finalize the notebook by removing all redundant or failed cells and organizing the functional code into a streamlined end-to-end pipeline.

## Fix Extraction and Setup Paths

### Subtask:
Initialize project directories and implement robust logic to extract reuploaded zip files while handling potential path or OS errors.


**Reasoning**:
I will consolidate the directory setup and implement robust zip extraction logic with error handling as requested in the subtask instructions.



In [ ]:
import os
import shutil

# 1. Define core directory variables
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# 2. Ensure directories exist on Google Drive
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

# 3. Map zip filenames to clean target folder names
zip_mapping = {
    'Pool Table Detection.v6i.yolov8.zip': 'Pool-Table-Detection-6',
    'CueTor Billiards Training.v8i.yolov8.zip': 'CueTor-Billiards-Training-8',
    'pool-balls-yolov11m.v14i.yolov8.zip': 'pool-balls-yolov11m-14'
}

print('--- Starting Shell-based Extraction Process ---')

# 4. Loop to extract each zip file using shell unzip for robustness
for zip_name, target_folder in zip_mapping.items():
    zip_path = os.path.join('/content', zip_name)

    if not os.path.exists(zip_path):
        print(f'Skipping: {zip_name} (File not found in /content)')
        continue

    try:
        temp_extract_path = os.path.join('/content', f'temp_{target_folder}')
        if os.path.exists(temp_extract_path):
            shutil.rmtree(temp_extract_path)
        os.makedirs(temp_extract_path, exist_ok=True)

        print(f'Extracting {zip_name} via shell...')
        # Use !unzip via os.system for better compatibility with large files/drive mount
        exit_code = os.system(f'unzip -q "{zip_path}" -d "{temp_extract_path}"')

        if exit_code != 0:
            raise OSError(f"Unzip failed with exit code {exit_code}")

        # Move to persistent location on Google Drive
        drive_target_path = os.path.join(DATASETS_BASE, target_folder)
        if os.path.exists(drive_target_path):
            shutil.rmtree(drive_target_path)

        shutil.move(temp_extract_path, drive_target_path)
        print(f'✅ Successfully persisted to: {drive_target_path}')

    except Exception as e:
        print(f'❌ Error processing {zip_name}: {e}')

print('--- Extraction Process Complete ---')

# Task
Generate the original user task based on the user message history and the plan.
If you see that the previous request was stopped, you MUST ignore the previous plan.
If the plan involves accessing or using specific data files or URLs, include
their full path(s) or URL(s) enclosed in double quotes in the task.
Be concise and reflect the final goal in the natural language. The output should be a fully self-contained task.
# Plan:
* **Robust Shell Extraction**: Implement a robust extraction cell using shell commands (!unzip) to bypass the Python-specific OS errors and verify the contents of the zip file before attempting to move them to Google Drive.
* **Final Task**: Confirm the successful extraction and persistence of the dataset to your Drive.


# Task
Implement a robust dataset extraction and persistence pipeline for the billiards pocket detector project. Specifically, use shell-based extraction commands (like `unzip`) to handle the zip files located at "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip". The process must extract these files into temporary directories in `/content` and then move the resulting folders to their persistent locations on Google Drive within "/content/drive/MyDrive/billiards_training/datasets/". Ensure the script includes error handling to manage failed extractions or missing files, and verify that the data is correctly persisted to Drive before proceeding.

## Robust Shell Extraction

### Subtask:
Implement a robust extraction cell using shell commands to bypass OS errors and persist data to Google Drive.


**Reasoning**:
I will implement a robust extraction cell using shell commands (unzip) to bypass the previously encountered OSError [Errno 22] and ensure the datasets are moved to persistent storage on Google Drive.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

# 1. Define mapping between zip files and target Drive directory names
zip_mapping = {
    '/content/Pool Table Detection.v6i.yolov8.zip': 'Pool-Table-Detection-6',
    '/content/CueTor Billiards Training.v8i.yolov8.zip': 'CueTor-Billiards-Training-8',
    '/content/pool-balls-yolov11m.v14i.yolov8.zip': 'pool-balls-yolov11m-14'
}

# Base directory on Google Drive
DRIVE_DATASETS_BASE = '/content/drive/MyDrive/billiards_training/datasets/'
os.makedirs(DRIVE_DATASETS_BASE, exist_ok=True)

print('--- Starting Robust Shell Extraction ---')

for zip_path, target_name in zip_mapping.items():
    # 3. Check if zip file exists
    if not os.path.exists(zip_path):
        print(f'Skipping: {zip_path} not found.')
        continue

    try:
        temp_extract_dir = f'/content/temp_extract_{target_name}'
        if os.path.exists(temp_extract_dir):
            shutil.rmtree(temp_extract_dir)
        os.makedirs(temp_extract_dir, exist_ok=True)

        # 4. Use shell unzip to bypass OSError [Errno 22]
        print(f'Extracting {os.path.basename(zip_path)}...')
        exit_code = os.system(f'unzip -q "{zip_path}" -d "{temp_extract_dir}"')

        if exit_code != 0:
            print(f'❌ Failed to unzip {zip_path}. Exit code: {exit_code}')
            continue

        # 5. Define final persistent path
        final_drive_path = os.path.join(DRIVE_DATASETS_BASE, target_name)

        if os.path.exists(final_drive_path):
            shutil.rmtree(final_drive_path)

        # Move contents to Drive
        shutil.move(temp_extract_dir, final_drive_path)

        # 7. Print confirmation
        print(f'✅ Successfully persisted {target_name} to: {final_drive_path}')

    except Exception as e:
        # 6. Catch and report errors for individual files
        print(f'❌ Error processing {zip_path}: {str(e)}')

print('--- Extraction Process Complete ---')

# Task
Implement a robust dataset extraction and persistence pipeline for the billiards pocket detector project. Specifically, use shell-based extraction commands (like `unzip`) to handle the zip files located at "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip". The process must extract these files into temporary directories in `/content` and then move the resulting folders to their persistent locations on Google Drive within "/content/drive/MyDrive/billiards_training/datasets/". Ensure the script includes error handling to manage failed extractions or missing files, and verify that the data is correctly persisted to Drive before proceeding.

## Consolidate Setup and Paths

### Subtask:
Initialize the project environment by installing dependencies, mounting Google Drive, and defining global path variables.


**Reasoning**:
I will create a single consolidated code block to install dependencies, mount Google Drive, and initialize the project directory structure as specified in the instructions.



In [ ]:
!pip install -q ultralytics roboflow kagglehub

import os
import shutil
import yaml
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define global path variables
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# Ensure all defined directories exist
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

print(f'Setup complete.')
print(f'PROJECT_DIR: {PROJECT_DIR}')
print(f'DATASETS_BASE: {DATASETS_BASE}')
print(f'COMBINED_DIR: {COMBINED_DIR}')

# Task
# Task
Implement a robust dataset extraction and persistence pipeline for the billiards pocket detector project. Specifically, use shell-based extraction commands (like `unzip`) to handle the zip files located at "/content/Pool Table Detection.v6i.yolov8.zip", "/content/CueTor Billiards Training.v8i.yolov8.zip", and "/content/pool-balls-yolov11m.v14i.yolov8.zip". The process must extract these files into temporary directories in `/content` and then move the resulting folders to their persistent locations on Google Drive within "/content/drive/MyDrive/billiards_training/datasets/". Ensure the script includes error handling to manage failed extractions or missing files, and verify that the data is correctly persisted to Drive before proceeding.

## Unify Path Definitions

### Subtask:
Initialize the project environment by installing dependencies, mounting Google Drive, and defining authoritative global path variables.


**Reasoning**:
I will consolidate the environment setup by installing necessary libraries, mounting Google Drive, and defining the authoritative global path variables in a single code block to ensure consistency across the project.



In [24]:
!pip install -q ultralytics roboflow kagglehub

import os
import shutil
import yaml
from google.colab import drive

# 3. Mount Google Drive
drive.mount('/content/drive')

# 4. Define authoritative global path variables
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# 5. Create directories if they do not exist
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATASETS_BASE, exist_ok=True)
os.makedirs(COMBINED_DIR, exist_ok=True)

# 6. Print verification
print(f'PROJECT_DIR: {PROJECT_DIR}')
print(f'DATASETS_BASE: {DATASETS_BASE}')
print(f'COMBINED_DIR: {COMBINED_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR: /content/drive/MyDrive/billiards_training
DATASETS_BASE: /content/drive/MyDrive/billiards_training/datasets
COMBINED_DIR: /content/drive/MyDrive/billiards_training/datasets/combined_dataset


## Map All Drive and Local Datasets

### Subtask:
Scan local and Drive directories to identify all dataset folders for merging.


**Reasoning**:
I will create a Python script to scan both the local and Drive directories to find and store the paths of the four required YOLO dataset folders by checking for the presence of 'data.yaml' files.



In [25]:
import os

# 1. Define the directories to scan
search_paths = [
    '/content',
    '/content/drive/MyDrive/billiards_training/datasets'
]

# 2. Keywords for the specific datasets we need
required_keywords = ['pool-balls', 'Pool-Table-Detection', 'CueTor', 'cue-detat']

source_datasets = []

print('--- Scanning for Valid YOLO Datasets ---')

for base_path in search_paths:
    if not os.path.exists(base_path):
        continue

    # Iterate through folders in the base path
    for item in os.listdir(base_path):
        full_path = os.path.join(base_path, item)

        if os.path.isdir(full_path):
            # 3. Check if it's one of the required datasets and has a data.yaml
            is_required = any(keyword.lower() in item.lower() for keyword in required_keywords)
            has_yaml = os.path.exists(os.path.join(full_path, 'data.yaml'))

            if is_required and has_yaml:
                if full_path not in source_datasets:
                    source_datasets.append(full_path)
                    print(f'✅ Found valid dataset: {full_path}')

# 4. Final verification
print('\n--- Summary ---')
print(f'Total datasets found: {len(source_datasets)}')
for ds in source_datasets:
    print(f'- {ds}')

if len(source_datasets) < 4:
    print('\n⚠️ Warning: Fewer than 4 datasets found. Please ensure all sources are downloaded/extracted.')

--- Scanning for Valid YOLO Datasets ---
✅ Found valid dataset: /content/temp_extract_pool-balls-yolov11m-14
✅ Found valid dataset: /content/CueTor_Billiards_Training.v8i.yolov8
✅ Found valid dataset: /content/temp_pool-balls-yolov11m-14
✅ Found valid dataset: /content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6
✅ Found valid dataset: /content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8
✅ Found valid dataset: /content/drive/MyDrive/billiards_training/datasets/pool-balls-yolov11m-14

--- Summary ---
Total datasets found: 6
- /content/temp_extract_pool-balls-yolov11m-14
- /content/CueTor_Billiards_Training.v8i.yolov8
- /content/temp_pool-balls-yolov11m-14
- /content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6
- /content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8
- /content/drive/MyDrive/billiards_training/datasets/pool-balls-yolov11m-14


## Execute Consolidated Merge

### Subtask:
Merge all identified YOLO datasets into a single master dataset at `COMBINED_DIR` and generate a unified `data.yaml` file.


**Reasoning**:
I will implement a consolidated merge script that iterates through the identified source datasets, maps their local classes to a global master list, and copies images and updated labels into the `COMBINED_DIR` while generating a unified `data.yaml`.



In [26]:
import os
import shutil
import yaml

# 1. Prioritize specific paths from the previously scanned source_datasets
# We want the persistent Drive versions and the Kaggle source for cue-detat
preferred_sources = [
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    '/content/drive/MyDrive/billiards_training/datasets/pool-balls-yolov11m-14',
    '/kaggle/input/cue-detat'
]

# Filter to ensure we only use existing paths
final_sources = [p for p in preferred_sources if os.path.exists(p)]

def consolidated_merge(source_paths, target_path):
    master_classes = []
    # Setup target structure
    for split in ['train', 'valid', 'test']:
        os.makedirs(os.path.join(target_path, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(target_path, split, 'labels'), exist_ok=True)

    print(f'Merging {len(source_paths)} datasets into {target_path}...')

    for folder in source_paths:
        y_path = os.path.join(folder, 'data.yaml')
        if not os.path.exists(y_path):
            continue

        with open(y_path, 'r') as f:
            cfg = yaml.safe_load(f)
            names = cfg.get('names', [])
            if isinstance(names, dict):
                names = [names[i] for i in sorted(names.keys())]

            # Map local indices to master indices
            class_map = {}
            for i, name in enumerate(names):
                if name not in master_classes:
                    master_classes.append(name)
                class_map[i] = master_classes.index(name)

        for split in ['train', 'valid', 'test']:
            # Handle variations in split naming (e.g., 'val' vs 'valid')
            src_split = split
            if not os.path.exists(os.path.join(folder, split)):
                if split == 'valid' and os.path.exists(os.path.join(folder, 'val')):
                    src_split = 'val'
                else:
                    continue

            s_img = os.path.join(folder, src_split, 'images')
            s_lbl = os.path.join(folder, src_split, 'labels')

            if os.path.exists(s_img):
                for f in os.listdir(s_img):
                    shutil.copy2(os.path.join(s_img, f), os.path.join(target_path, split, 'images', f))

            if os.path.exists(s_lbl):
                for f in os.listdir(s_lbl):
                    with open(os.path.join(s_lbl, f), 'r') as lf:
                        lines = lf.readlines()
                    with open(os.path.join(target_path, split, 'labels', f), 'a') as lf:
                        for l in lines:
                            parts = l.split()
                            if parts:
                                parts[0] = str(class_map[int(parts[0])])
                                lf.write(' '.join(parts) + '\n')

    # Generate unified data.yaml
    new_cfg = {
        'train': os.path.join(target_path, 'train', 'images'),
        'val': os.path.join(target_path, 'valid', 'images'),
        'test': os.path.join(target_path, 'test', 'images'),
        'nc': len(master_classes),
        'names': master_classes
    }
    with open(os.path.join(target_path, 'data.yaml'), 'w') as f:
        yaml.dump(new_cfg, f)

    return master_classes

# Execute merge
master_classes = consolidated_merge(final_sources, COMBINED_DIR)

# Verify statistics
print('\n--- Merge Statistics ---')
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(COMBINED_DIR, split, 'images')
    count = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'{split.capitalize()} images: {count}')

print(f'\nUnified Master Classes: {master_classes}')

Merging 4 datasets into /content/drive/MyDrive/billiards_training/datasets/combined_dataset...

--- Merge Statistics ---
Train images: 28374
Valid images: 6188
Test images: 3641

Unified Master Classes: ['pool-table', 'pool-table-hole', 'pool-table-side', '0', '1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9', 'solid', 'striped']


## Dynamic Class Indexing

### Subtask:
Identify the 'pool-table-hole' index from the master class list to set POCKET_CLASS_IDX for the training phase.


**Reasoning**:
I will add a code block to dynamically find the index of 'pool-table-hole' in the master_classes list and assign it to POCKET_CLASS_IDX as per the instructions.



In [27]:
# Find the index for 'pool-table-hole' dynamically from the master_classes list
try:
    pocket_idx = master_classes.index('pool-table-hole')
    POCKET_CLASS_IDX = [pocket_idx]
    print(f"Success: 'pool-table-hole' found at index {pocket_idx}")
    print(f"POCKET_CLASS_IDX set to: {POCKET_CLASS_IDX}")
except ValueError:
    # Fallback or error reporting if the class is missing
    print("Error: 'pool-table-hole' not found in master_classes.")
    print(f"Current master_classes: {master_classes}")
    POCKET_CLASS_IDX = [1] # Common default for this dataset if mapping failed
    print(f"Defaulting POCKET_CLASS_IDX to: {POCKET_CLASS_IDX}")

Success: 'pool-table-hole' found at index 1
POCKET_CLASS_IDX set to: [1]


## Notebook Cleanup

### Subtask:
Remove previous failed extraction attempts, redundant setup cells, and duplicate training logic to create a functional and clean pipeline.


# Task
# Task
Verify the integrity of the merged dataset in `COMBINED_DIR` (located at "/content/drive/MyDrive/billiards_training/datasets/combined_dataset") and confirm it is ready for YOLOv8 training. This involves:
1. Verifying the directory structure to ensure "train", "valid", and "test" splits exist alongside the master "data.yaml".
2. Counting the image and label files within each split to ensure parity and successful data transfer from the four source datasets.
3. Inspecting the contents of the generated "data.yaml" (at "/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml") to confirm the class names (specifically the 'pool-table-hole' index), class counts, and relative paths are correctly defined.
4. Summarizing the dataset readiness for the subsequent training phase.

## Verify Directory Structure

### Subtask:
List the directories and files within `COMBINED_DIR` to confirm the presence of train, valid, and test splits along with the master `data.yaml`.


**Reasoning**:
I will add a code block to verify the directory structure of the combined dataset by listing its contents and checking for the required split folders and the data.yaml file.



In [3]:
import os

# Redefine path variables to ensure they are in scope
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# 1. Check if COMBINED_DIR exists
if os.path.exists(COMBINED_DIR):
    print(f'--- Verifying Directory Structure in: {COMBINED_DIR} ---')

    # 2. List top-level contents
    contents = os.listdir(COMBINED_DIR)
    print(f'Top-level contents: {contents}')

    # 3. Verify core components
    expected_components = ['train', 'valid', 'test', 'data.yaml']
    for comp in expected_components:
        exists = comp in contents or (comp == 'valid' and 'val' in contents)
        print(f"- {comp}: {'✅ Found' if exists else '❌ Missing'}")

    # 4. Check subdirectories in each split
    for split in ['train', 'valid', 'test']:
        split_path = os.path.join(COMBINED_DIR, split)
        # Handle variations like 'val' instead of 'valid'
        if not os.path.exists(split_path) and split == 'valid':
            split_path = os.path.join(COMBINED_DIR, 'val')

        if os.path.exists(split_path):
            sub_contents = os.listdir(split_path)
            has_images = 'images' in sub_contents
            has_labels = 'labels' in sub_contents
            print(f'\n[{split.upper()}] split subdirectories:')
            print(f'  - images: {"✅ OK" if has_images else "❌ Missing"}')
            print(f'  - labels: {"✅ OK" if has_labels else "❌ Missing"}')
else:
    print(f'Error: COMBINED_DIR not found at {COMBINED_DIR}')

--- Verifying Directory Structure in: /content/drive/MyDrive/billiards_training/datasets/combined_dataset ---
Top-level contents: ['train', 'valid', 'test', 'data.yaml']
- train: ✅ Found
- valid: ✅ Found
- test: ✅ Found
- data.yaml: ✅ Found

[TRAIN] split subdirectories:
  - images: ✅ OK
  - labels: ✅ OK

[VALID] split subdirectories:
  - images: ✅ OK
  - labels: ✅ OK

[TEST] split subdirectories:
  - images: ✅ OK
  - labels: ✅ OK


## Check Image and Label Counts

### Subtask:
Count the number of image and label files in each split directory to ensure they match and verify the data transfer was successful.


**Reasoning**:
I will create a code block that counts the image and label files for each dataset split to ensure data parity and verify the success of the merge.



In [4]:
import os

def count_files(directory, extensions):
    """Counts files in a directory that match the given extensions."""
    if not os.path.exists(directory):
        return 0
    return len([f for f in os.listdir(directory) if f.lower().endswith(extensions)])

# Define extensions for images and labels
img_exts = ('.jpg', '.jpeg', '.png')
lbl_exts = ('.txt')

splits = ['train', 'valid', 'test']

print('--- File Count Verification ---')
for split in splits:
    img_dir = os.path.join(COMBINED_DIR, split, 'images')
    lbl_dir = os.path.join(COMBINED_DIR, split, 'labels')

    num_images = count_files(img_dir, img_exts)
    num_labels = count_files(lbl_dir, lbl_exts)

    status = "✅ Match" if num_images == num_labels else "❌ Mismatch"

    print(f'[{split.upper()}]')
    print(f'  Images: {num_images}')
    print(f'  Labels: {num_labels}')
    print(f'  Status: {status}')

    if num_images != num_labels:
        diff = abs(num_images - num_labels)
        print(f'  ⚠️ Warning: Discrepancy of {diff} file(s) found in {split} split.')
    print()

--- File Count Verification ---
[TRAIN]
  Images: 18246
  Labels: 3648
  Status: ❌ Mismatch
  ⚠️ Warning: Discrepancy of 14598 file(s) found in train split.

[VALID]
  Images: 524
  Labels: 524
  Status: ✅ Match

[TEST]
  Images: 262
  Labels: 262
  Status: ✅ Match



## Inspect Master data.yaml

### Subtask:
Read and verify the contents of the generated `data.yaml` to confirm class mapping, counts, and directory paths.


**Reasoning**:
I will add a code block to read and parse the `data.yaml` file from the `COMBINED_DIR`, verifying class names, counts, and directory paths as per the subtask instructions.



In [5]:
import yaml
import os

# 1. Define the path to data.yaml
yaml_path = os.path.join(COMBINED_DIR, 'data.yaml')

print(f'--- Inspecting Master Config: {yaml_path} ---')

# 2. Open and load the YAML file
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # 3. Print entire content for manual inspection
    print('Full YAML Content:')
    print(yaml.dump(config, default_flow_style=False))

    # 4. Specifically verify class names, counts, and paths
    names = config.get('names', [])
    nc = config.get('nc', 0)
    train_path = config.get('train')
    val_path = config.get('val')
    test_path = config.get('test')

    print('\n--- Key Verifications ---')
    print(f'Classes (names): {names}')
    print(f'Class Count (nc): {nc}')
    print(f'Train Path: {train_path}')
    print(f'Val Path:   {val_path}')
    print(f'Test Path:  {test_path}')

    # 5. Search for pool-table-hole index
    if 'pool-table-hole' in names:
        pocket_idx = names.index('pool-table-hole')
        print(f"\n✅ 'pool-table-hole' found at index: {pocket_idx}")
        print(f"Matches POCKET_CLASS_IDX logic: {pocket_idx == POCKET_CLASS_IDX[0]}")
    else:
        print("\n❌ Error: 'pool-table-hole' not found in names list.")

    # 6. Check consistency between nc and names length
    if nc == len(names):
        print(f"✅ Consistency Check: 'nc' ({nc}) matches length of 'names' ({len(names)}).")
    else:
        print(f"❌ Consistency Check Failed: 'nc' ({nc}) does not match length of 'names' ({len(names)}).")
else:
    print(f'Error: {yaml_path} not found.')

--- Inspecting Master Config: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml ---
Full YAML Content:
names:
- pool-table
- pool-table-hole
- pool-table-side
nc: 3
test: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/test/images
train: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images
val: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/valid/images


--- Key Verifications ---
Classes (names): ['pool-table', 'pool-table-hole', 'pool-table-side']
Class Count (nc): 3
Train Path: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images
Val Path:   /content/drive/MyDrive/billiards_training/datasets/combined_dataset/valid/images
Test Path:  /content/drive/MyDrive/billiards_training/datasets/combined_dataset/test/images

✅ 'pool-table-hole' found at index: 1


NameError: name 'POCKET_CLASS_IDX' is not defined

**Reasoning**:
The previous cell failed because 'POCKET_CLASS_IDX' was not defined in the current scope. I will redefine it by searching for 'pool-table-hole' in the 'names' list from the loaded YAML configuration to ensure the verification logic can complete.



In [6]:
import yaml
import os

# 1. Define the path to data.yaml
yaml_path = os.path.join(COMBINED_DIR, 'data.yaml')

print(f'--- Inspecting Master Config: {yaml_path} ---')

# 2. Open and load the YAML file
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # 3. Print entire content for manual inspection
    print('Full YAML Content:')
    print(yaml.dump(config, default_flow_style=False))

    # 4. Specifically verify class names, counts, and paths
    names = config.get('names', [])
    nc = config.get('nc', 0)
    train_path = config.get('train')
    val_path = config.get('val')
    test_path = config.get('test')

    print('\n--- Key Verifications ---')
    print(f'Classes (names): {names}')
    print(f'Class Count (nc): {nc}')
    print(f'Train Path: {train_path}')
    print(f'Val Path:   {val_path}')
    print(f'Test Path:  {test_path}')

    # 5. Search for pool-table-hole index and define POCKET_CLASS_IDX
    if 'pool-table-hole' in names:
        pocket_idx = names.index('pool-table-hole')
        POCKET_CLASS_IDX = [pocket_idx]
        print(f"\n✅ 'pool-table-hole' found at index: {pocket_idx}")
        print(f"POCKET_CLASS_IDX defined as: {POCKET_CLASS_IDX}")
    else:
        print("\n❌ Error: 'pool-table-hole' not found in names list.")

    # 6. Check consistency between nc and names length
    if nc == len(names):
        print(f"✅ Consistency Check: 'nc' ({nc}) matches length of 'names' ({len(names)}).")
    else:
        print(f"❌ Consistency Check Failed: 'nc' ({nc}) does not match length of 'names' ({len(names)}).")
else:
    print(f'Error: {yaml_path} not found.')

--- Inspecting Master Config: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml ---
Full YAML Content:
names:
- pool-table
- pool-table-hole
- pool-table-side
nc: 3
test: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/test/images
train: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images
val: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/valid/images


--- Key Verifications ---
Classes (names): ['pool-table', 'pool-table-hole', 'pool-table-side']
Class Count (nc): 3
Train Path: /content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images
Val Path:   /content/drive/MyDrive/billiards_training/datasets/combined_dataset/valid/images
Test Path:  /content/drive/MyDrive/billiards_training/datasets/combined_dataset/test/images

✅ 'pool-table-hole' found at index: 1
POCKET_CLASS_IDX defined as: [1]
✅ Consistency Check: 'nc' (3) matches length of 'names' (3).


## Final Task

### Subtask:
Summarize the findings from the dataset integrity checks and provide a final verdict on training readiness.


## Summary:

### Q&A

**Is the combined dataset ready for YOLOv8 training?**
No. While the directory structure and `data.yaml` configuration are correct, a critical integrity issue was found in the training split where the number of images ($18,246$) significantly exceeds the number of label files ($3,648$).

**Are the class mappings and paths correctly defined in the configuration?**
Yes. The `data.yaml` file correctly identifies three classes: `pool-table`, `pool-table-hole` (at index 1), and `pool-table-side`. All paths point to the correct absolute locations in Google Drive.

---

### Data Analysis Key Findings

*   **Directory Integrity:** The top-level structure is verified. The `train`, `valid`, and `test` directories all contain the necessary `images` and `labels` subfolders.
*   **Split Parity:**
    *   **Validation Set:** Perfect match with $524$ images and $524$ labels.
    *   **Test Set:** Perfect match with $262$ images and $262$ labels.
    *   **Training Set:** Critical mismatch detected. There are $18,246$ images but only $3,648$ labels, leaving $14,598$ images without corresponding annotation files.
*   **Configuration Accuracy:** The `data.yaml` file is properly formatted for YOLOv8, with `nc: 3` and the `pool-table-hole` class correctly indexed at position 1.

---

### Insights or Next Steps

*   **Investigate Data Merge:** The discrepancy in the training split suggests a failure during the file copy or merging process. You must re-run the dataset aggregation logic specifically for the training labels to ensure all $14,598$ missing files are accounted for.
*   **Sanity Check Image-Label Mapping:** Before training, perform a check to ensure that the $3,648$ existing labels actually correspond to the correct images (i.e., they share the same filenames), as a partial copy often indicates a deeper synchronization error.


# Task
# Task
Address the critical data integrity issue in the combined dataset and proceed with YOLOv8 training. First, fix the training label merge to ensure all 18,246 images in "/content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images" have corresponding label files in the labels folder, resolving the identified mismatch of 14,598 missing files. Once data integrity is verified, initialize and run a pausable YOLOv8n training session using the master "data.yaml" at "/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml", saving checkpoints to Google Drive every 5 epochs. After training, validate the best model's performance (mAP50) against the validation split. Finally, export the trained model to TFLite FP16 format, saving it to the "/content/drive/MyDrive/billiards_training/exports" directory, and clean the notebook of any redundant cells.

## Fix Training Label Merge

### Subtask:
Re-run the dataset aggregation logic to resolve the training label mismatch of 14,598 missing files.


**Reasoning**:
I will implement a refined merge script that specifically targets the training labels across all four source datasets. This script will ensure the correct class mapping is applied and will fill in the missing 14,598 label files to achieve image-label parity in the combined training directory.



In [7]:
import os
import shutil
import yaml

# Configuration paths
COMBINED_DIR = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset'
sources = [
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    '/content/drive/MyDrive/billiards_training/datasets/pool-balls-yolov11m-14',
    '/kaggle/input/cue-detat'
]

# Load Master Class List from existing data.yaml
yaml_path = os.path.join(COMBINED_DIR, 'data.yaml')
with open(yaml_path, 'r') as f:
    master_cfg = yaml.safe_load(f)
master_classes = master_cfg['names']

print(f'Master Classes: {master_classes}')
target_train_lbl = os.path.join(COMBINED_DIR, 'train', 'labels')
os.makedirs(target_train_lbl, exist_ok=True)

for folder in sources:
    if not os.path.exists(folder): continue

    # Get local class mapping
    y_path = os.path.join(folder, 'data.yaml')
    if not os.path.exists(y_path): continue

    with open(y_path, 'r') as f:
        cfg = yaml.safe_load(f)
        local_names = cfg.get('names', [])
        if isinstance(local_names, dict): local_names = [local_names[i] for i in sorted(local_names.keys())]
        class_map = {i: master_classes.index(name) for i, name in enumerate(local_names) if name in master_classes}

    # Process only the training labels
    src_split = 'train' if os.path.exists(os.path.join(folder, 'train')) else 'val' if os.path.exists(os.path.join(folder, 'val')) else None
    if not src_split: continue

    s_lbl = os.path.join(folder, src_split, 'labels')
    if os.path.exists(s_lbl):
        print(f'Fixing labels from: {folder}...')
        for f in os.listdir(s_lbl):
            if not f.endswith('.txt'): continue
            target_f_path = os.path.join(target_train_lbl, f)

            with open(os.path.join(s_lbl, f), 'r') as lf: lines = lf.readlines()
            with open(target_f_path, 'w') as lf: # Use 'w' to ensure clean rewrite with correct mapping
                for l in lines:
                    parts = l.split()
                    if parts and int(parts[0]) in class_map:
                        parts[0] = str(class_map[int(parts[0])])
                        lf.write(' '.join(parts) + '\n')

# Validation Check
img_count = len([f for f in os.listdir(os.path.join(COMBINED_DIR, 'train', 'images')) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
lbl_count = len([f for f in os.listdir(target_train_lbl) if f.endswith('.txt')])

print('\n--- Final Validation ---')
print(f'Training Images: {img_count}')
print(f'Training Labels: {lbl_count}')

if img_count == lbl_count == 18246:
    print('✅ Success: Image-to-label parity achieved at 18,246 files!')
else:
    print(f'⚠ Still Mismatched: {img_count} images vs {lbl_count} labels.')

Master Classes: ['pool-table', 'pool-table-hole', 'pool-table-side']
Fixing labels from: /content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6...
Fixing labels from: /content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8...

--- Final Validation ---
Training Images: 18246
Training Labels: 3648
⚠ Still Mismatched: 18246 images vs 3648 labels.


**Reasoning**:
The previous attempt to fix the label mismatch failed because 'pool-balls-yolov11m-14' and 'cue-detat' (via kagglehub) were not correctly processed due to subfolder structures or path inconsistencies. I will re-implement the fix with more robust directory searching for 'data.yaml' and 'labels' in each source to ensure all 18,246 training labels are found and copied.



In [8]:
import os
import shutil
import yaml
import kagglehub

# 1. Update/Verify Kaggle path
kaggle_path = kagglehub.dataset_download('hereliesaz/cue-detat')

# 2. Source directories to search
sources = [
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    '/content/drive/MyDrive/billiards_training/datasets/pool-balls-yolov11m-14',
    kaggle_path
]

# Target directory
TARGET_LBL_DIR = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/labels'
os.makedirs(TARGET_LBL_DIR, exist_ok=True)

# Master class list for mapping
master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']

print('--- Starting Robust Label Recovery ---')

for base_folder in sources:
    if not os.path.exists(base_folder):
        print(f'Missing source: {base_folder}')
        continue

    # Walk through the folder to find data.yaml and corresponding labels
    found_data = False
    for root, dirs, files in os.walk(base_folder):
        if 'data.yaml' in files:
            y_path = os.path.join(root, 'data.yaml')
            with open(y_path, 'r') as f:
                cfg = yaml.safe_load(f)
                local_names = cfg.get('names', [])
                if isinstance(local_names, dict):
                    local_names = [local_names[i] for i in sorted(local_names.keys())]

                # Create class map for existing master classes
                class_map = {i: master_classes.index(name) for i, name in enumerate(local_names) if name in master_classes}

            # Now look for 'train/labels' relative to this data.yaml or in the same root
            # We check both the current root/train/labels and root/labels
            potential_lbl_dirs = [
                os.path.join(root, 'train', 'labels'),
                os.path.join(root, 'labels')
            ]

            for lbl_dir in potential_lbl_dirs:
                if os.path.exists(lbl_dir):
                    print(f'Processing labels from: {lbl_dir}')
                    for f_name in os.listdir(lbl_dir):
                        if not f_name.endswith('.txt'): continue

                        with open(os.path.join(lbl_dir, f_name), 'r') as lf: lines = lf.readlines()
                        with open(os.path.join(TARGET_LBL_DIR, f_name), 'w') as lf_out:
                            for line in lines:
                                parts = line.split()
                                if parts and int(parts[0]) in class_map:
                                    parts[0] = str(class_map[int(parts[0])])
                                    lf_out.write(' '.join(parts) + '\n')
                    found_data = True
            if found_data: break # Stop searching this base_folder if data found

# Final Validation
img_dir = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images'
img_count = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
lbl_count = len([f for f in os.listdir(TARGET_LBL_DIR) if f.endswith('.txt')])

print('\n--- Final Statistics ---')
print(f'Total Training Images: {img_count}')
print(f'Total Training Labels: {lbl_count}')

if img_count == lbl_count:
    print('✅ Success: Parity achieved!')
else:
    print(f'⚠ Still Mismatched: Difference of {abs(img_count - lbl_count)} files.')

100%|██████████| 1.52G/1.52G [00:15<00:00, 103MB/s] 

Extracting files...


--- Starting Robust Label Recovery ---
Processing labels from: /content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6/train/labels
Processing labels from: /content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8/train/labels
Processing labels from: /root/.cache/kagglehub/datasets/hereliesaz/cue-detat/versions/1/CueTor Billiards Training.v8i.yolov8/train/labels

--- Final Statistics ---
Total Training Images: 18246
Total Training Labels: 3648
⚠ Still Mismatched: Difference of 14598 files.


**Reasoning**:
The previous attempts failed to achieve label parity because the 'pool-balls-yolov11m-14' dataset, which likely contains the majority of the images, was missing its label mapping logic or proper path identification. I will implement a final, exhaustive label recovery script that specifically targets every image in the training directory, finds its matching source label from the four persistent datasets, and applies the correct class mapping for the 'pool-table-hole' project.



In [9]:
import os
import yaml

# 1. Define paths
TRAIN_IMG_DIR = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/images'
TARGET_LBL_DIR = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset/train/labels'
os.makedirs(TARGET_LBL_DIR, exist_ok=True)

sources = [
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    '/content/drive/MyDrive/billiards_training/datasets/pool-balls-yolov11m-14',
    '/root/.cache/kagglehub/datasets/hereliesaz/cue-detat/versions/1'
]

master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']

# 2. Map all available source labels to a lookup dictionary
print('Indexing source labels...')
label_lookup = {}  # filename -> (full_path, class_map)

for base in sources:
    if not os.path.exists(base): continue

    current_class_map = {}
    # Search for data.yaml to define class mapping
    for root, _, files in os.walk(base):
        if 'data.yaml' in files:
            with open(os.path.join(root, 'data.yaml'), 'r') as f:
                cfg = yaml.safe_load(f)
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
                current_class_map = {i: master_classes.index(n) for i, n in enumerate(names) if n in master_classes}

            # Index all .txt files in this source subtree
            for r, _, f_list in os.walk(root):
                if 'labels' in r:
                    for f_name in f_list:
                        if f_name.endswith('.txt'):
                            label_lookup[f_name] = (os.path.join(r, f_name), current_class_map)

# 3. Iterate through combined images and copy/remap matching labels
print('Synching labels to images...')
images = [f for f in os.listdir(TRAIN_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
processed_count = 0

for img_name in images:
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    if lbl_name in label_lookup:
        src_path, cmap = label_lookup[lbl_name]
        with open(src_path, 'r') as f_in:
            lines = f_in.readlines()

        with open(os.path.join(TARGET_LBL_DIR, lbl_name), 'w') as f_out:
            for line in lines:
                parts = line.split()
                if parts and int(parts[0]) in cmap:
                    parts[0] = str(cmap[int(parts[0])])
                    f_out.write(' '.join(parts) + '\n')
        processed_count += 1

# 4. Final Validation
img_count = len(images)
lbl_count = len(os.listdir(TARGET_LBL_DIR))
print(f'\n--- Recovery Results ---')
print(f'Total Training Images: {img_count}')
print(f'Total Training Labels: {lbl_count}')

if img_count == lbl_count:
    print('✅ Success: Absolute parity achieved!')
else:
    print(f'⚠ Still Mismatched: {img_count - lbl_count} labels missing.')

Indexing source labels...
Synching labels to images...

--- Recovery Results ---
Total Training Images: 18246
Total Training Labels: 18246
✅ Success: Absolute parity achieved!


**Reasoning**:
The training dataset integrity has been verified with 18,246 images and 18,246 matching labels. I will now proceed to configure and execute the YOLOv8n training session using the combined dataset and saving checkpoints to Google Drive.



In [10]:
from ultralytics import YOLO
import os

# 1. Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = os.path.join(COMBINED_DIR, 'data.yaml')
TRAINING_NAME = 'pocket_detector'

# 2. Define checkpoint path for resume logic
checkpoint_path = os.path.join(PROJECT_DIR, TRAINING_NAME, 'weights', 'last.pt')
resume_training = os.path.exists(checkpoint_path)

# 3. Load model (resume from last.pt if exists, else start fresh with yolov8n.pt)
if resume_training:
    print(f'Resuming training from checkpoint: {checkpoint_path}')
    model = YOLO(checkpoint_path)
else:
    print('Starting fresh training with yolov8n.pt')
    model = YOLO('yolov8n.pt')

# 4. Execute training
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT_DIR,
    name=TRAINING_NAME,
    save=True,
    save_period=5,
    resume=resume_training,
    device=0
)

print(f'Training session concluded. Weights saved to: {results.save_dir}')

ModuleNotFoundError: No module named 'ultralytics'

**Reasoning**:
The previous cell failed because the 'ultralytics' library was not installed in the current session. I will install the required 'ultralytics' library using pip to resolve the ModuleNotFoundError before attempting to run the training code.



In [11]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 45.8/112.6 GB disk)


**Reasoning**:
With `ultralytics` installed and dataset integrity confirmed (18,246 images and labels), I will now execute the YOLOv8n training session. The model is configured for 50 epochs with checkpoints saved to Drive every 5 epochs to support pausing and resuming.



In [ ]:
from ultralytics import YOLO
import os

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = os.path.join(COMBINED_DIR, 'data.yaml')
TRAINING_NAME = 'pocket_detector'

# Define checkpoint path for resume logic
checkpoint_path = os.path.join(PROJECT_DIR, TRAINING_NAME, 'weights', 'last.pt')
resume_training = os.path.exists(checkpoint_path)

# Load model (resume from last.pt if exists, else start fresh with yolov8n.pt)
if resume_training:
    print(f'Resuming training from checkpoint: {checkpoint_path}')
    model = YOLO(checkpoint_path)
else:
    print('Starting fresh training with yolov8n.pt')
    model = YOLO('yolov8n.pt')

# Execute training with periodic checkpointing
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT_DIR,
    name=TRAINING_NAME,
    save=True,
    save_period=5,
    resume=resume_training,
    device=0
)

print(f'Training session concluded. Weights saved to: {results.save_dir}')

Resuming training from checkpoint: /content/drive/MyDrive/billiards_training/pocket_detector/weights/last.pt
Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive